In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------
# Step 1: Set working directory to the Trademarkia project
# ---------------------------------------------------------

import os

# Path to project folder in Google Drive
PROJECT_PATH = "/content/drive/MyDrive/trademarkia_project"

# Create the folder if it does not already exist
os.makedirs(PROJECT_PATH, exist_ok=True)

# Change current working directory to the project folder
os.chdir(PROJECT_PATH)

print("Working directory successfully set to:")
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------
# Step 2: Import required libraries
# ---------------------------------------------------------

import os
import re
import zipfile

import pandas as pd
from pathlib import Path

In [ ]:
# ---------------------------------------------------------
# Step 3: Verify dataset file exists in project directory
# ---------------------------------------------------------

DATASET_PATH = "/content/drive/MyDrive/trademarkia_project/twenty+newsgroups.zip"

if os.path.exists(DATASET_PATH):
    print("Dataset found:")
    print(DATASET_PATH)
else:
    raise FileNotFoundError(
        "Dataset not found. Please upload twenty+newsgroups.zip to the trademarkia_project folder."
    )

In [ ]:
# ---------------------------------------------------------
# Step 4: Extract the Twenty Newsgroups dataset
# ---------------------------------------------------------

# Dataset location (stored in Google Drive)
DATASET_ZIP_PATH = "/content/drive/MyDrive/trademarkia_project/twenty+newsgroups.zip"

# Directory where dataset will be extracted
EXTRACT_PATH = "/content/drive/MyDrive/trademarkia_project/twenty_newsgroups"

# Create extraction directory if it does not exist
os.makedirs(EXTRACT_PATH, exist_ok=True)

# Extract the dataset
with zipfile.ZipFile(DATASET_ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_PATH)

print("Dataset extracted successfully.")
print("Extraction directory:", EXTRACT_PATH)

In [ ]:
# ---------------------------------------------------------
# Step 5: Inspect extracted dataset structure
# ---------------------------------------------------------

EXTRACT_PATH = "/content/drive/MyDrive/trademarkia_project/twenty_newsgroups"

# List top-level items in the extracted dataset folder
top_level_items = os.listdir(EXTRACT_PATH)

print("Top-level items inside extracted dataset folder:")
print(top_level_items[:20])

In [ ]:
# ---------------------------------------------------------
# Step 6: Extract the Twenty Newsgroups TAR dataset
# ---------------------------------------------------------

import tarfile

EXTRACT_PATH = "/content/drive/MyDrive/trademarkia_project/twenty_newsgroups"

tar_file_path = os.path.join(EXTRACT_PATH, "20_newsgroups.tar.gz")

with tarfile.open(tar_file_path, "r:gz") as tar:
    tar.extractall(EXTRACT_PATH)

print("TAR dataset extracted successfully.")

In [ ]:
# ---------------------------------------------------------
# Step 7: Define dataset root containing category folders
# ---------------------------------------------------------

DATASET_ROOT = "/content/drive/MyDrive/trademarkia_project/twenty_newsgroups/20_newsgroups"

if os.path.exists(DATASET_ROOT):
    print("Dataset root located at:")
    print(DATASET_ROOT)
else:
    raise FileNotFoundError("Dataset root folder not found after extraction.")

In [ ]:
# ---------------------------------------------------------
# Step 8: Verify category folders in dataset
# ---------------------------------------------------------

DATASET_ROOT = "/content/drive/MyDrive/trademarkia_project/twenty_newsgroups/20_newsgroups"

categories = sorted(os.listdir(DATASET_ROOT))

print("Total categories:", len(categories))
print("Category folders:")
print(categories)

In [ ]:
# ---------------------------------------------------------
# Step 9: Load all documents into a structured dataframe
# ---------------------------------------------------------

import os
import pandas as pd

DATASET_ROOT = "/content/drive/MyDrive/trademarkia_project/twenty_newsgroups/20_newsgroups"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/trademarkia_project/raw_documents.csv"

records = []
doc_id = 0

# Iterate through category folders in sorted order for reproducibility
for category in sorted(os.listdir(DATASET_ROOT)):
    category_path = os.path.join(DATASET_ROOT, category)

    if not os.path.isdir(category_path):
        continue

    # Iterate through files in sorted order for deterministic document IDs
    for filename in sorted(os.listdir(category_path)):
        file_path = os.path.join(category_path, filename)

        if not os.path.isfile(file_path):
            continue

        try:
            # Latin-1 is commonly used for the 20 Newsgroups dataset
            with open(file_path, "r", encoding="latin1") as file:
                text = file.read()

            records.append(
                {
                    "doc_id": doc_id,
                    "category": category,
                    "filename": filename,
                    "file_path": file_path,
                    "raw_text": text,
                }
            )
            doc_id += 1

        except Exception as error:
            # Log unreadable files without stopping the pipeline
            print(f"Warning: Could not read file {file_path}")
            print(f"Reason: {error}")

# Create dataframe
df = pd.DataFrame(records)

# Save raw loaded dataset for traceability and reuse
df.to_csv(OUTPUT_CSV_PATH, index=False)

print("Total documents loaded:", len(df))
print("Saved raw document dataframe to:", OUTPUT_CSV_PATH)

df.head()

In [ ]:
# ---------------------------------------------------------
# Step 10: Inspect dataset structure and basic statistics
# ---------------------------------------------------------

print("Dataset shape (rows, columns):", df.shape)

print("\nColumns in dataset:")
print(list(df.columns))

print("\nCategory distribution:")
print(df["category"].value_counts())

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nSample records:")
display(df.head())

In [ ]:
# ---------------------------------------------------------
# Step 11: Inspect a sample raw document
# ---------------------------------------------------------

import random

sample_index = random.randint(0, len(df) - 1)

print("Sample index:", sample_index)
print("Category:", df.loc[sample_index, "category"])
print("Filename:", df.loc[sample_index, "filename"])

print("\nPreview of raw document (first 1500 characters):\n")
print(df.loc[sample_index, "raw_text"][:1500])

In [ ]:
# ---------------------------------------------------------
# Step 12: Define text cleaning function
# ---------------------------------------------------------

import re

def clean_newsgroup_text(text: str) -> str:
    """
    Clean raw text from the 20 Newsgroups dataset.

    This function removes:
    - email/news headers
    - quoted replies
    - email addresses
    - URLs
    - long separators
    - unusual symbols
    - excessive whitespace

    Parameters
    ----------
    text : str
        Raw document text.

    Returns
    -------
    str
        Cleaned text.
    """

    if not isinstance(text, str):
        return ""

    # Remove common email/news headers
    text = re.sub(
        r'(?im)^(from|subject|organization|lines|distribution|keywords|date|path|xref|message-id|sender):.*$',
        '',
        text
    )

    # Remove quoted replies (lines starting with ">")
    text = re.sub(r'(?m)^>.*$', '', text)

    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove long separators
    text = re.sub(r'[-_=]{2,}', ' ', text)

    # Remove unusual symbols while keeping basic punctuation
    text = re.sub(r'[^a-zA-Z0-9\s\.,!?;:()\'"-]', ' ', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# ---------------------------------------------------------
# Step 13: Apply text cleaning to the dataset
# ---------------------------------------------------------

# Apply cleaning function
df["clean_text"] = df["raw_text"].apply(clean_newsgroup_text)

print("Text cleaning applied successfully.")

# Remove rows where cleaning produced empty text
initial_count = len(df)
df = df[df["clean_text"].str.strip() != ""]

final_count = len(df)

print(f"Removed {initial_count - final_count} empty documents after cleaning.")
print(f"Remaining documents: {final_count}")

# Save cleaned dataset for later pipeline stages
CLEAN_DATASET_PATH = "/content/drive/MyDrive/trademarkia_project/cleaned_documents.csv"
df.to_csv(CLEAN_DATASET_PATH, index=False)

print("Cleaned dataset saved to:", CLEAN_DATASET_PATH)

# Preview cleaned results
df[["doc_id", "category", "clean_text"]].head()

In [ ]:
# ---------------------------------------------------------
# Step 14: Compare raw vs cleaned text samples
# ---------------------------------------------------------

import random

sample_indices = random.sample(range(len(df)), 3)

for idx in sample_indices:
    print("Category:", df.loc[idx, "category"])

    print("\nRAW TEXT (first 500 chars):\n")
    print(df.loc[idx, "raw_text"][:500])

    print("\nCLEANED TEXT (first 500 chars):\n")
    print(df.loc[idx, "clean_text"][:500])

    print("\n" + "-" * 100 + "\n")

In [ ]:
# ---------------------------------------------------------
# Step 15: Remove very short documents
# ---------------------------------------------------------

# Calculate word count of cleaned text
df["clean_length"] = df["clean_text"].apply(lambda x: len(x.split()))

initial_count = len(df)

# Keep documents with at least 10 words
df = df[df["clean_length"] >= 10].reset_index(drop=True)

final_count = len(df)

print(f"Documents removed: {initial_count - final_count}")
print(f"Documents remaining after filtering: {final_count}")

# Save updated cleaned dataset
FILTERED_DATASET_PATH = "/content/drive/MyDrive/trademarkia_project/filtered_documents.csv"
df.to_csv(FILTERED_DATASET_PATH, index=False)

print("Filtered dataset saved to:", FILTERED_DATASET_PATH)

df.head()

In [ ]:
# ---------------------------------------------------------
# Step 16: Install required libraries for embeddings and search
# ---------------------------------------------------------

# Sentence Transformers for semantic embeddings
# FAISS for efficient vector similarity search

!pip install -q sentence-transformers faiss-cpu

In [ ]:
# ---------------------------------------------------------
# Step 17: Load Sentence Transformer embedding model
# ---------------------------------------------------------

from sentence_transformers import SentenceTransformer

# Lightweight, high-quality semantic embedding model
MODEL_NAME = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

print(f"Embedding model '{MODEL_NAME}' loaded successfully.")

In [ ]:
# ---------------------------------------------------------
# Step 18: Generate semantic embeddings for documents
# ---------------------------------------------------------

import numpy as np

texts = df["clean_text"].tolist()

# Generate embeddings using the transformer model
embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

# Normalize embeddings for cosine similarity search
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print("Embeddings generated successfully.")
print("Embedding matrix shape:", embeddings.shape)

# Save embeddings for reuse
EMBEDDINGS_PATH = "/content/drive/MyDrive/trademarkia_project/document_embeddings.npy"
np.save(EMBEDDINGS_PATH, embeddings)

print("Embeddings saved to:", EMBEDDINGS_PATH)

In [ ]:
# ---------------------------------------------------------
# Step 19: Create FAISS vector index for semantic search
# ---------------------------------------------------------

import faiss
import numpy as np

dimension = embeddings.shape[1]

# Use Inner Product index (works as cosine similarity with normalized vectors)
index = faiss.IndexFlatIP(dimension)

# Add embeddings to the index
index.add(embeddings)

print("FAISS index created successfully.")
print("Total vectors stored in index:", index.ntotal)

# Save FAISS index for reuse
FAISS_INDEX_PATH = "/content/drive/MyDrive/trademarkia_project/faiss_index.bin"
faiss.write_index(index, FAISS_INDEX_PATH)

print("FAISS index saved to:", FAISS_INDEX_PATH)

In [ ]:
# ---------------------------------------------------------
# Step 20: Test semantic search using FAISS index
# ---------------------------------------------------------

query = "space shuttle launch"

# Generate query embedding
query_vector = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

# Normalize query vector (required for cosine similarity)
query_vector = query_vector / np.linalg.norm(query_vector, axis=1, keepdims=True)

k = 5  # number of results to retrieve

distances, indices = index.search(query_vector, k)

print("Query:", query)
print("\nTop results:\n")

for i, idx in enumerate(indices[0]):
    print(f"Result {i+1}")
    print("Category:", df.iloc[idx]["category"])
    print("Text:", df.iloc[idx]["clean_text"][:300])
    print("-" * 60)

In [ ]:
# ---------------------------------------------------------
# Step 21: Install library for fuzzy clustering
# ---------------------------------------------------------

# scikit-fuzzy provides Fuzzy C-Means clustering
# which allows documents to belong to multiple clusters

!pip install -q scikit-fuzzy

In [ ]:
# ---------------------------------------------------------
# Step 22: Import fuzzy clustering library
# ---------------------------------------------------------

import skfuzzy as fuzz

In [ ]:
# ---------------------------------------------------------
# Step 23: Prepare embeddings for fuzzy clustering
# ---------------------------------------------------------

# Fuzzy C-Means expects data in shape: (features, samples)
# Our embeddings are currently: (samples, features)

embedding_matrix = embeddings.T

print("Embedding matrix prepared for fuzzy clustering.")

print("Original embedding shape (samples, features):", embeddings.shape)
print("Clustering input shape (features, samples):", embedding_matrix.shape)

In [ ]:
# ---------------------------------------------------------
# Step 24: Run Fuzzy C-Means clustering on embeddings
# ---------------------------------------------------------

n_clusters = 20  # initial cluster count (can be tuned later)

cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    embedding_matrix,
    c=n_clusters,
    m=2,
    error=0.005,
    maxiter=1000,
    init=None
)

print("Fuzzy clustering completed.")
print("Number of clusters:", n_clusters)
print("Membership matrix shape:", u.shape)
print("Fuzzy Partition Coefficient (FPC):", fpc)

# Save cluster outputs
CLUSTER_CENTER_PATH = "/content/drive/MyDrive/trademarkia_project/cluster_centers.npy"
MEMBERSHIP_PATH = "/content/drive/MyDrive/trademarkia_project/cluster_membership.npy"

np.save(CLUSTER_CENTER_PATH, cntr)
np.save(MEMBERSHIP_PATH, u)

print("Cluster centers saved to:", CLUSTER_CENTER_PATH)
print("Membership matrix saved to:", MEMBERSHIP_PATH)

In [ ]:
# ---------------------------------------------------------
# Step 25: Assign dominant cluster to each document
# ---------------------------------------------------------

# For each document, find the cluster with the highest membership score
dominant_clusters = np.argmax(u, axis=0)

df["dominant_cluster"] = dominant_clusters

print("Dominant cluster assigned to each document.")

# Show sample results
display(df[["doc_id", "category", "dominant_cluster"]].head())

# Show cluster distribution
print("\nCluster distribution:")
print(df["dominant_cluster"].value_counts().sort_index())

# Save dataset with cluster labels
CLUSTERED_DATASET_PATH = "/content/drive/MyDrive/trademarkia_project/documents_with_clusters.csv"
df.to_csv(CLUSTERED_DATASET_PATH, index=False)

print("\nClustered dataset saved to:", CLUSTERED_DATASET_PATH)

In [ ]:
# ---------------------------------------------------------
# Step 26: Initialize semantic cache
# ---------------------------------------------------------

# Dictionary to store cached query results
# Format:
# { query_text : { "response": ..., "cluster": ..., "embedding": ... } }

semantic_cache = {}

# Cache statistics for monitoring performance
cache_stats = {
    "total_entries": 0,
    "hit_count": 0,
    "miss_count": 0
}

print("Semantic cache initialized.")
print("Current cache size:", len(semantic_cache))

In [ ]:
# ---------------------------------------------------------
# Step 27: Define cosine similarity function for cache lookup
# ---------------------------------------------------------

import numpy as np

def compute_similarity(vec1, vec2):
    """
    Compute cosine similarity between two normalized vectors.
    Since embeddings are normalized, cosine similarity = dot product.
    """
    return float(np.dot(vec1, vec2))

print("Similarity function ready.")

In [ ]:
# ---------------------------------------------------------
# Step 28: Configure semantic cache similarity threshold
# ---------------------------------------------------------

# Minimum cosine similarity required to treat a query as a cache hit
# Higher value = stricter match
# Lower value = more aggressive caching

SIMILARITY_THRESHOLD = 0.85

print(f"Semantic cache similarity threshold set to: {SIMILARITY_THRESHOLD}")

In [ ]:
# ---------------------------------------------------------
# Step 29: Define function to check semantic cache
# ---------------------------------------------------------

def check_semantic_cache(query_embedding, query_cluster):
    """
    Check whether a semantically similar query exists in the cache.
    """

    best_match = None
    best_score = 0

    for cached_query, cache_entry in semantic_cache.items():

        # Optional cluster filtering (faster lookup)
        if cache_entry["cluster"] != query_cluster:
            continue

        cached_embedding = cache_entry["embedding"]

        score = compute_similarity(query_embedding, cached_embedding)

        if score > best_score:
            best_score = score
            best_match = cached_query

    if best_score >= SIMILARITY_THRESHOLD:

        cache_stats["hit_count"] += 1

        return {
            "cache_hit": True,
            "matched_query": best_match,
            "similarity_score": best_score,
            "result": semantic_cache[best_match]["result"],
            "dominant_cluster": semantic_cache[best_match]["cluster"]
        }

    cache_stats["miss_count"] += 1

    return {"cache_hit": False}

In [ ]:
# ---------------------------------------------------------
# Step 30: Define function to store results in semantic cache
# ---------------------------------------------------------

def store_in_cache(query, query_embedding, result_text, dominant_cluster):
    """
    Store a query and its result in the semantic cache.
    """

    # Avoid overwriting existing cache entries
    if query in semantic_cache:
        return

    semantic_cache[query] = {
        "embedding": query_embedding,
        "result": result_text,
        "cluster": dominant_cluster
    }

    # Update cache statistics
    cache_stats["total_entries"] = len(semantic_cache)

In [ ]:
# ---------------------------------------------------------
# Step 31: Define main query processing function
# ---------------------------------------------------------

def process_query(query):
    """
    Process a user query using semantic cache + FAISS retrieval.
    """

    # Generate query embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )[0]

    # Normalize query embedding for cosine similarity
    query_embedding = query_embedding / np.linalg.norm(query_embedding)

    # Predict query cluster using similarity to cluster centers
    cluster_similarities = np.dot(cntr, query_embedding)
    query_cluster = int(np.argmax(cluster_similarities))

    # Check semantic cache first
    cache_result = check_semantic_cache(query_embedding, query_cluster)

    if cache_result["cache_hit"]:
        return {
            "query": query,
            **cache_result
        }

    # Perform semantic search using FAISS
    query_vector = np.array([query_embedding], dtype=np.float32)
    distances, indices = index.search(query_vector, 1)

    doc_index = int(indices[0][0])

    result_text = df.iloc[doc_index]["clean_text"]
    dominant_cluster = int(df.iloc[doc_index]["dominant_cluster"])

    # Store result in semantic cache
    store_in_cache(query, query_embedding, result_text, dominant_cluster)

    return {
        "query": query,
        "cache_hit": False,
        "similarity_score": None,
        "matched_query": None,
        "result": result_text,
        "dominant_cluster": dominant_cluster
    }

In [ ]:
# ---------------------------------------------------------
# Step 32: Test semantic cache behavior
# ---------------------------------------------------------

test_query = "space shuttle launch"

print("First query (expected cache miss):")
response1 = process_query(test_query)
print(response1)

print("\nSecond query (expected cache hit):")
response2 = process_query(test_query)
print(response2)

print("\nCache statistics:")
print(cache_stats)

In [ ]:
# ---------------------------------------------------------
# Step 33: Test semantically similar query
# ---------------------------------------------------------

similar_query = "launch of the space shuttle"

print("Testing semantically similar query:")
response = process_query(similar_query)

print("\nResponse:")
print(response)

print("\nUpdated cache statistics:")
print(cache_stats)

In [ ]:
# ---------------------------------------------------------
# Step 34: Define function to retrieve cache statistics
# ---------------------------------------------------------

def get_cache_stats():
    """
    Return semantic cache statistics.
    """

    total_requests = cache_stats["hit_count"] + cache_stats["miss_count"]

    hit_rate = (
        cache_stats["hit_count"] / total_requests
        if total_requests > 0
        else 0
    )

    return {
        "total_entries": cache_stats["total_entries"],
        "hit_count": cache_stats["hit_count"],
        "miss_count": cache_stats["miss_count"],
        "hit_rate": round(hit_rate, 3)
    }

In [ ]:
# ---------------------------------------------------------
# Step 35: Define function to clear the semantic cache
# ---------------------------------------------------------

def clear_cache():
    """
    Reset the semantic cache and cache statistics.
    """

    semantic_cache.clear()

    cache_stats["total_entries"] = 0
    cache_stats["hit_count"] = 0
    cache_stats["miss_count"] = 0

    return {"message": "Semantic cache cleared successfully."}

In [ ]:
# ---------------------------------------------------------
# Step 36: Install FastAPI and server dependencies
# ---------------------------------------------------------

!pip install -q fastapi uvicorn pyngrok

In [ ]:
# ---------------------------------------------------------
# Step 37: Import FastAPI framework
# ---------------------------------------------------------

from fastapi import FastAPI
from pydantic import BaseModel

In [ ]:
# ---------------------------------------------------------
# Step 38: Create FastAPI application
# ---------------------------------------------------------

app = FastAPI(
    title="Semantic Search with Cluster-Aware Cache",
    description="API for semantic document retrieval using embeddings, FAISS search, fuzzy clustering, and semantic caching.",
    version="1.0"
)

In [ ]:
# ---------------------------------------------------------
# Step 39: Define request model for query endpoint
# ---------------------------------------------------------

class QueryRequest(BaseModel):
    """
    Request body for semantic search queries.
    """
    query: str

In [ ]:
# ---------------------------------------------------------
# Step 40: Create query endpoint
# ---------------------------------------------------------

@app.post("/query")
def query_endpoint(request: QueryRequest):
    """
    Process a semantic search query using cache + FAISS retrieval.
    """

    query_text = request.query
    response = process_query(query_text)

    return response

In [ ]:
# ---------------------------------------------------------
# Step 41: Create cache statistics endpoint
# ---------------------------------------------------------

@app.get("/cache/stats")
def cache_stats_endpoint():
    """
    Return statistics about the semantic cache.
    """

    return get_cache_stats()

In [ ]:
# ---------------------------------------------------------
# Step 42: Create endpoint to clear semantic cache
# ---------------------------------------------------------

@app.delete("/cache")
def clear_cache_endpoint():
    """
    Reset the semantic cache and statistics.
    """
    return clear_cache()

In [ ]:
#---------------------------
# Step 43 : app.py (Full Semantic Cache Implementation)
#---------------------------
%%writefile /content/drive/MyDrive/trademarkia-semantic-search/app/app.py

import faiss
import numpy as np
import pandas as pd
from pathlib import Path
from fastapi import FastAPI
from pydantic import BaseModel, Field
from sentence_transformers import SentenceTransformer

PROJECT_DIR = Path("/content/drive/MyDrive/trademarkia-semantic-search")
DOCUMENTS_PATH = PROJECT_DIR / "data/processed/documents_with_clusters.csv"
FAISS_INDEX_PATH = PROJECT_DIR / "data/artifacts/faiss_index.bin"
CLUSTER_CENTERS_PATH = PROJECT_DIR / "data/artifacts/cluster_centers.npy"

MODEL_NAME = "all-MiniLM-L6-v2"
SIMILARITY_THRESHOLD = 0.85

app = FastAPI(title="Trademarkia Semantic Cache API")

class QueryRequest(BaseModel):
    query: str = Field(..., min_length=1)

# Global state for loaded models and data
df, index, cluster_centers, embedding_model = None, None, None, None
semantic_cache = {}
cache_stats = {"total_entries": 0, "hit_count": 0, "miss_count": 0}

@app.on_event("startup")
def startup_event():
    global df, index, cluster_centers, embedding_model
    df = pd.read_csv(DOCUMENTS_PATH)
    index = faiss.read_index(str(FAISS_INDEX_PATH))
    cluster_centers = np.load(CLUSTER_CENTERS_PATH).astype(np.float32)
    embedding_model = SentenceTransformer(MODEL_NAME)

@app.get("/")
def home(): return {"message": "Trademarkia semantic search API running"}

@app.post("/query")
def query_endpoint(request: QueryRequest):
    query = request.query

    # 1. Generate and normalize query embedding
    q_emb = embedding_model.encode([query], convert_to_numpy=True)[0].astype(np.float32)
    q_emb = q_emb / np.linalg.norm(q_emb)

    # 2. Predict cluster
    similarities = np.dot(cluster_centers, q_emb)
    query_cluster = int(np.argmax(similarities))

    # 3. Check Semantic Cache
    best_match, best_score = None, 0
    for cached_q, entry in semantic_cache.items():
        if entry["cluster"] == query_cluster:
            score = np.dot(q_emb, entry["embedding"])
            if score > best_score:
                best_score, best_match = score, cached_q

    if best_match and best_score >= SIMILARITY_THRESHOLD:
        cache_stats["hit_count"] += 1
        return {
            "query": query,
            "cache_hit": True,
            "matched_query": best_match,
            "similarity_score": float(best_score),
            "result": semantic_cache[best_match]["result"],
            "dominant_cluster": query_cluster
        }

    # 4. Cache Miss: FAISS Search
    cache_stats["miss_count"] += 1
    distances, indices = index.search(np.array([q_emb]), 1)
    doc_idx = int(indices[0][0])
    result_text = str(df.iloc[doc_idx]["clean_text"])

    # 5. Store in Cache
    semantic_cache[query] = {
        "embedding": q_emb,
        "result": result_text,
        "cluster": query_cluster
    }
    cache_stats["total_entries"] = len(semantic_cache)

    return {
        "query": query,
        "cache_hit": False,
        "matched_query": None,
        "similarity_score": float(distances[0][0]),
        "result": result_text,
        "dominant_cluster": query_cluster
    }

@app.get("/cache/stats")
def get_stats():
    total = cache_stats["hit_count"] + cache_stats["miss_count"]
    hit_rate = cache_stats["hit_count"] / total if total > 0 else 0
    return {
        "total_entries": cache_stats["total_entries"],
        "hit_count": cache_stats["hit_count"],
        "miss_count": cache_stats["miss_count"],
        "hit_rate": round(hit_rate, 3)
    }

@app.delete("/cache")
def clear_cache_endpoint():
    semantic_cache.clear()
    cache_stats["total_entries"] = 0
    cache_stats["hit_count"] = 0
    cache_stats["miss_count"] = 0
    return {"message": "Semantic cache cleared successfully"}

In [ ]:
# ---------------------------------------------------------
# Step 44: Create requirements.txt for project dependencies
# ---------------------------------------------------------

%%writefile /content/drive/MyDrive/trademarkia_project/requirements.txt
fastapi
uvicorn
numpy
pandas
faiss-cpu
sentence-transformers
scikit-fuzzy
pydantic

In [ ]:
# ---------------------------------------------------------
# Step 45: Verify required project artifacts
# ---------------------------------------------------------

import os

project_path = "/content/drive/MyDrive/trademarkia-semantic-search"

required_files = {
    "documents_with_clusters.csv": "data/processed/documents_with_clusters.csv",
    "faiss_index.bin": "data/artifacts/faiss_index.bin",
    "cluster_centers.npy": "data/artifacts/cluster_centers.npy",
    "app.py": "app/app.py",
    "requirements.txt": "requirements.txt"
}

print("Checking required project files...\n")

missing_files = []

for name, relative_path in required_files.items():

    file_path = os.path.join(project_path, relative_path)

    if os.path.exists(file_path):
        print(f"✓ Found: {name}")
    else:
        print(f"✗ Missing: {name}")
        missing_files.append(name)

if len(missing_files) == 0:
    print("\nAll required project files are present.")
else:
    print("\nWarning: Some required files are missing.")

In [ ]:
#-------------------------------------------------
# STEP - 46:  Restart server to apply cache logic
#-------------------------------------------------
!pkill -f uvicorn
os.chdir("/content/drive/MyDrive/trademarkia-semantic-search")
!nohup uvicorn app.app:app --host 0.0.0.0 --port 8000 > nohup.out 2>&1 &
time.sleep(5)
print("Server restarted with full Semantic Cache logic enabled.")

In [ ]:
#verifying whether the server is running or not
!curl http://localhost:8000

In [ ]:
# ---------------------------------------------------------
# Check Server Logs for Initialization Progress
# ---------------------------------------------------------

import os

log_path = "/content/drive/MyDrive/trademarkia-semantic-search/nohup.out"
if os.path.exists(log_path):
    print("Checking last 20 lines of server logs...")
    !tail -n 20 "{log_path}"
else:
    print("Log file not found yet. The server process might still be starting.")

In [ ]:
# ---------------------------------------------------------
# Step 47: Install and configure ngrok
# ---------------------------------------------------------

# Install ngrok library
!pip install pyngrok

from pyngrok import ngrok
import os

print("ngrok installed successfully.")

In [ ]:
# ---------------------------------------------------------
# Step 47.1: Install correct ngrok authentication token
# ---------------------------------------------------------

#!ngrok config add-authtoken UR_AUTH_TOKEN (token removed for safety)

In [ ]:
# ---------------------------------------------------------
# Step 47.2: Create public tunnel for FastAPI
# ---------------------------------------------------------

from pyngrok import ngrok

public_url = ngrok.connect(8000)

print("Public API URL:", public_url)

In [ ]:
# ---------------------------------------------------------
# Step 47.3: Display public API URL and verification links
# ---------------------------------------------------------

print("\nPublic API URL:")
print(public_url)

print("\nSwagger API Documentation:")
print(f"{public_url}/docs")

print("\nOpenAPI Schema:")
print(f"{public_url}/openapi.json")

In [ ]:
# ---------------------------------------------------------
# Step 48.1: Install requests library
# ---------------------------------------------------------

!pip install requests

In [ ]:
# ---------------------------------------------------------
# Step 48.2: Import libraries for API testing
# ---------------------------------------------------------

import requests
import json

In [ ]:
# ---------------------------------------------------------
# Step 48.3: Define API base URL correctly
# ---------------------------------------------------------

# Extract the public URL string from the ngrok tunnel object
API_BASE_URL = public_url.public_url

print("Testing API at:")
print(API_BASE_URL)

In [ ]:
# ---------------------------------------------------------
# Step 48.4: Robust Test for Root Endpoint with Retries
# ---------------------------------------------------------

import requests
import time

def wait_for_server(url, timeout=60):
    print(f"Waiting for server at {url} to initialize (timeout: {timeout}s)...")
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                print("Server is UP and running!")
                return response
            else:
                print(f"Server returned status {response.status_code}. Retrying...")
        except Exception:
            # Connection errors are expected while the port is not yet bound
            pass
        time.sleep(5)
    return None

# Construct the root URL
root_url = f"{API_BASE_URL}/"
response = wait_for_server(root_url)

if response:
    print("\nRoot Endpoint Response:")
    print(response.json())
else:
    print("\nServer failed to start within the timeout period. Please check nohup.out for errors.")

In [ ]:
# ---------------------------------------------------------
# Step 48.5: Test semantic search query endpoint
# ---------------------------------------------------------

query_payload = {
    "query": "space shuttle launch"
}

response = requests.post(
    f"{API_BASE_URL}/query",
    json=query_payload
)

print("Query Response:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# ---------------------------------------------------------
# Step 48.6: Test cache statistics endpoint
# ---------------------------------------------------------

response = requests.get(f"{API_BASE_URL}/cache/stats")

print("Cache Stats:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# ---------------------------------------------------------
# Step 48.7: Test cache reset endpoint
# ---------------------------------------------------------

response = requests.delete(f"{API_BASE_URL}/cache")

print("Cache Clear Response:")
print(response.json())

In [ ]:
# ---------------------------------------------------------
# Step 48.5: Test semantic search query endpoint
# ---------------------------------------------------------

query_payload = {
    "query": "space shuttle launch"
}

response = requests.post(
    f"{API_BASE_URL}/query",
    json=query_payload
)

print("Query Response:")
import json
print(json.dumps(response.json(), indent=2))

In [ ]:
# ---------------------------------------------------------
# Step 48.6: Test cache statistics endpoint
# ---------------------------------------------------------

response = requests.get(f"{API_BASE_URL}/cache/stats")

print("Cache Stats:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# ---------------------------------------------------------
# Debug API response
# ---------------------------------------------------------

print("Status Code:", response.status_code)
print("Raw Response:")
print(response.text)

In [ ]:
# ---------------------------------------------------------
# Step 48.5: Test semantic search query endpoint
# ---------------------------------------------------------

query_payload = {
    "query": "space shuttle launch"
}

response = requests.post(
    f"{API_BASE_URL}/query",
    json=query_payload
)

print("Query Response:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# ---------------------------------------------------------
# Step 48.6: Test cache statistics endpoint
# ---------------------------------------------------------

response = requests.get(f"{API_BASE_URL}/cache/stats")

print("Cache Stats:")
print(json.dumps(response.json(), indent=2))

In [ ]:
# ---------------------------------------------------------
# Step 48.7: Test cache reset endpoint
# ---------------------------------------------------------

response = requests.delete(f"{API_BASE_URL}/cache")

print("Cache Clear Response:")
print(response.json())

In [ ]:
# ---------------------------------------------------------
# Step 49 :  ReadMe file
# ---------------------------------------------------------
%%writefile /content/drive/MyDrive/trademarkia-semantic-search/README.md

# Trademarkia Semantic Search with Cluster-Aware Semantic Cache

## Project Overview

This project implements a **production-style semantic search system** using modern Natural Language Processing and vector search techniques.
The system retrieves relevant documents using **sentence embeddings**, **FAISS vector indexing**, **fuzzy clustering**, and a **semantic caching mechanism** to improve query performance.

The API is built using **FastAPI** and exposed publicly via **ngrok** for testing.

---

# System Architecture

User Query → Sentence Embedding → Cluster Prediction (Fuzzy C-Means) → Semantic Cache Lookup → FAISS Vector Search (if cache miss) → Return Result + Store in Cache.

---

# Repository Structure

```text
trademarkia-semantic-search/
├── app/
│   └── app.py                      # FastAPI application core logic
├── data/
│   ├── raw/
│   │   └── twenty+newsgroups.zip    # Original dataset archive
│   ├── processed/
│   │   ├── raw_documents.csv       # Initial loaded dataset
│   │   ├── cleaned_documents.csv   # Text-processed dataset
│   │   ├── filtered_documents.csv  # Final refined dataset
│   │   └── documents_with_clusters.csv # Clustered results
│   └── artifacts/
│       ├── document_embeddings.npy # Generated vector embeddings
│       ├── faiss_index.bin         # Efficient search index
│       ├── cluster_centers.npy     # Fuzzy C-Means centroids
│       └── cluster_membership.npy  # Membership scores
├── notebooks/
│   └── pipeline.ipynb              # Development and evaluation notebook
├── Dockerfile                      # Containerization instructions
├── docker-compose.yml              # Multi-container orchestration
├── requirements.txt                # Project dependencies
├── .gitignore                      # Files excluded from Git tracking
├── LICENSE                         # Project license/copyright notice
└── README.md                       # Project documentation
```

---

# Key Features

### Semantic Document Retrieval
Uses **SentenceTransformer embeddings** to capture semantic meaning rather than simple keyword matching.

### FAISS Vector Search
Efficient approximate nearest neighbor search for fast document retrieval across 20,000 documents.

### Fuzzy Clustering
Groups documents using **Fuzzy C-Means**, allowing for more nuanced semantic categorization.

### Cluster-Aware Semantic Cache
New queries first check the cache within their predicted clusters. This significantly reduces response time for repeated or semantically similar queries.

---

# Running the Project

## Install Dependencies
```bash
pip install -r requirements.txt
```

## Start the API Server
```bash
uvicorn app.app:app --host 0.0.0.0 --port 8000
```

---

# Docker Deployment

## Build Docker Image
```bash
docker build -t trademarkia-api .
```

## Run Docker Container
```bash
docker run -p 8000:8000 trademarkia-api
```

---

# API Documentation

FastAPI automatically generates documentation at:
`http://localhost:8000/docs`

---

# Author

**Syed Muskan**
B.Tech Computer Science (AI & ML) - PreFinal Year Student
VIT-AP UNIVERSITY


In [ ]:
# ---------------------------------------------------------
# Step 50: Create Dockerfile
# ---------------------------------------------------------

%%writefile /content/drive/MyDrive/trademarkia_project/Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]


In [ ]:
# ---------------------------------------------------------
# Step 51: Create docker-compose.yml
# ---------------------------------------------------------

%%writefile /content/drive/MyDrive/trademarkia_project/docker-compose.yml
version: '3.8'

services:
  trademarkia-api:
    build: .
    ports:
      - "8000:8000"

In [ ]:
# ---------------------------------------------------------
# Step 52: Verify project files
# ---------------------------------------------------------

import os

os.listdir("/content/drive/MyDrive/trademarkia_project")

In [ ]:
# ---------------------------------------------------------
# Step 53: Create simple copyright notice for assignment
# ---------------------------------------------------------

%%writefile /content/drive/MyDrive/trademarkia_project/LICENSE

Copyright (c) 2026 Syed Muskan

This repository was created as part of a technical evaluation.

All rights reserved.

The contents of this repository may be reviewed for recruitment
and evaluation purposes. No other rights are granted.

In [ ]:
%%writefile /content/drive/MyDrive/trademarkia_project/.gitignore
# -----------------------------
# Dataset archives
# -----------------------------
*.zip
*.tar.gz

# Extracted dataset folders
twenty_newsgroups/

# -----------------------------
# Python runtime files
# -----------------------------
__pycache__/
*.pyc

# Jupyter notebook checkpoints
.ipynb_checkpoints/

# -----------------------------
# Logs and temporary files
# -----------------------------
nohup.out
*.log

In [ ]:
# ---------------------------------------------------------
# Reorganize Trademarkia project structure in Google Drive
# ---------------------------------------------------------

import os
import shutil

base = "/content/drive/MyDrive/trademarkia_project"
repo = "/content/drive/MyDrive/trademarkia-semantic-search"

# Create main repo directory
os.makedirs(repo, exist_ok=True)

# Create folder structure
folders = [
    "app",
    "data/raw",
    "data/processed",
    "data/artifacts",
    "notebooks"
]

for f in folders:
    os.makedirs(os.path.join(repo, f), exist_ok=True)

# File move helper
def move_if_exists(src, dst):
    if os.path.exists(src):
        shutil.move(src, dst)
        print(f"Moved: {src} -> {dst}")
    else:
        print(f"Skipped (not found): {src}")

# ---------------------------------------------------------
# Move root files
# ---------------------------------------------------------

move_if_exists(f"{base}/README.md", f"{repo}/README.md")
move_if_exists(f"{base}/requirements.txt", f"{repo}/requirements.txt")
move_if_exists(f"{base}/Dockerfile", f"{repo}/Dockerfile")
move_if_exists(f"{base}/docker-compose.yml", f"{repo}/docker-compose.yml")
move_if_exists(f"{base}/.gitignore", f"{repo}/.gitignore")
move_if_exists(f"{base}/LICENSE", f"{repo}/LICENSE")

# ---------------------------------------------------------
# Move API
# ---------------------------------------------------------

move_if_exists(f"{base}/app.py", f"{repo}/app/app.py")

# ---------------------------------------------------------
# Move dataset
# ---------------------------------------------------------

move_if_exists(f"{base}/twenty+newsgroups.zip", f"{repo}/data/raw/twenty+newsgroups.zip")

# ---------------------------------------------------------
# Move processed datasets
# ---------------------------------------------------------

move_if_exists(f"{base}/raw_documents.csv", f"{repo}/data/processed/raw_documents.csv")
move_if_exists(f"{base}/cleaned_documents.csv", f"{repo}/data/processed/cleaned_documents.csv")
move_if_exists(f"{base}/filtered_documents.csv", f"{repo}/data/processed/filtered_documents.csv")
move_if_exists(f"{base}/documents_with_clusters.csv", f"{repo}/data/processed/documents_with_clusters.csv")

# ---------------------------------------------------------
# Move artifacts
# ---------------------------------------------------------

move_if_exists(f"{base}/document_embeddings.npy", f"{repo}/data/artifacts/document_embeddings.npy")
move_if_exists(f"{base}/faiss_index.bin", f"{repo}/data/artifacts/faiss_index.bin")
move_if_exists(f"{base}/cluster_centers.npy", f"{repo}/data/artifacts/cluster_centers.npy")
move_if_exists(f"{base}/cluster_membership.npy", f"{repo}/data/artifacts/cluster_membership.npy")

# ---------------------------------------------------------
# Move notebook
# ---------------------------------------------------------

for file in os.listdir(base):
    if file.endswith(".ipynb"):
        move_if_exists(f"{base}/{file}", f"{repo}/notebooks/{file}")

print("\nProject successfully reorganized.")
print("New repo location:", repo)

print("\nFinal structure:")
for root, dirs, files in os.walk(repo):
    level = root.replace(repo, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 4 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")